# Proximal Policy Optimization

The on-policy methods of Chapters 08 and 09 have a simple structure and few moving parts, but they left two problems open, and we saw both of them on Lunar Lander. The first is **sample efficiency**: every rollout is used for a single gradient step and then discarded, because the policy gradient is only valid for the policy that generated the data. The second is the **step size**: the actor is updated with a fixed learning rate in parameter space, but what matters is how much the *policy* changes, and the same parameter step can be harmless in one state and catastrophic in another. A single update that moves the policy too far produces bad data, and since the next update is computed from that data, the agent may never recover.

Chapter 10 solved the first problem by going off-policy, at the cost of a more delicate algorithm. This chapter stays on-policy and attacks both problems with a single idea: **a policy update should be as large as possible, but not so large that the data it was computed from stop being representative**. Making this idea precise leads to the notion of a **trust region** around the current policy, to the **TRPO** algorithm, and finally to [**Proximal Policy Optimization**](./papers/2017%20-%20Proximal%20Policy%20Optimization%20Algorithms.pdf), PPO, which obtains the same effect with a few lines of code and has become the most widely used reinforcement learning algorithm, from robot locomotion to video games to the fine-tuning of large language models.

The chapter proceeds as follows. We first look at the step-size problem more closely. We then ask what we can say about a *new* policy using data from the *old* one, which leads to the performance difference lemma, to a surrogate objective based on importance sampling, and to a bound that justifies the trust region. We sketch how TRPO enforces the trust region and why it is impractical, and derive PPO's clipped objective. We then implement PPO as a modification of the A2C code of Chapter 09, compare the two on Cart-Pole, and apply PPO to Lunar Lander, in both its discrete and continuous versions.

## The Step-Size Problem

Gradient ascent updates the parameters by $\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)$, a step of fixed size in **parameter space**. But the objective depends on the parameters only through the policy, and the relation between a change of $\theta$ and the resulting change of $\pi_\theta$ is far from uniform. Consider a softmax policy over two actions with a single preference parameter, $\pi_\theta(a_1 \mid s) = 1 / (1 + e^{-\theta})$. The same step $\Delta\theta = 1$ changes the probability of $a_1$ from $0.50$ to $0.73$ when $\theta = 0$, but only from $0.982$ to $0.993$ when $\theta = 4$. In a neural network, where every parameter affects the policy in every state, a step that is tiny in some states is huge in others, and the learning rate has to be chosen for the worst case.

Why is a large policy change dangerous, when in supervised learning a large step is at worst a wasted step? Because in reinforcement learning the **data depend on the policy**. If one update moves the policy to a region where it behaves badly, the next rollout is collected by that bad policy: the returns are poor, the advantages are noisy, and the gradient estimated from them points nowhere in particular. The agent has to climb back from a bad policy using bad data. Gradient clipping, which we used in Chapter 09, bounds the size of the parameter step; it does not bound the size of the policy change. The natural measure of the latter is the **Kullback-Leibler divergence** between the old and the new policy in a state,

$\displaystyle D_{\text{KL}}\big( \pi_{\theta_{\text{old}}}(\cdot \mid s)\ \big\|\ \pi_\theta(\cdot \mid s) \big) = \sum_a \pi_{\theta_{\text{old}}}(a \mid s) \log \frac{\pi_{\theta_{\text{old}}}(a \mid s)}{\pi_\theta(a \mid s)}$

which is zero when the two policies coincide and grows with their difference, in a way that does not depend on how the policies are parameterised. What we would like is an update that improves the objective while keeping the KL divergence, averaged over the states, below a threshold. To get there we first need a way to evaluate a candidate new policy without collecting data from it.

## Evaluating a New Policy with Old Data

### The Performance Difference Lemma

Suppose we have run the current policy $\pi$, estimated its value function $v_\pi$ and advantage function $a_\pi$, and we are considering a different policy $\pi'$. (In this section we write the objective as a function of the policy, $J(\pi)$, rather than of its parameters: $J(\pi_\theta)$ and $J(\theta)$ are the same quantity.) How much better is $\pi'$? The following identity, due to Kakade and Langford (2002), answers exactly this question. For any two policies,

$\displaystyle J(\pi') - J(\pi) = \mathbb{E}_{\tau \sim \pi'} \left[ \sum_{t=0}^{\infty} \gamma^{t}\, a_\pi(S_t, A_t) \right]$

The performance of $\pi'$ exceeds that of $\pi$ by the discounted sum of the **advantages of $\pi$** along the **trajectories of $\pi'$**. The proof is a telescoping argument. By the definition of the advantage and the relation between $q_\pi$ and $v_\pi$ of Chapter 03,

$\displaystyle a_\pi(s,a) = q_\pi(s,a) - v_\pi(s) = \mathbb{E}\big[ R_{t+1} + \gamma\, v_\pi(S_{t+1}) \mid S_t = s, A_t = a \big] - v_\pi(s)$

so that, along a trajectory generated by any policy, the expected advantage at step $t$ is the expected value of $R_{t+1} + \gamma v_\pi(S_{t+1}) - v_\pi(S_t)$. Summing with the discount factors,

$\displaystyle \mathbb{E}_{\tau \sim \pi'} \left[ \sum_{t=0}^{\infty} \gamma^{t}\, a_\pi(S_t, A_t) \right] = \mathbb{E}_{\tau \sim \pi'} \left[ \sum_{t=0}^{\infty} \gamma^{t} R_{t+1} + \sum_{t=0}^{\infty} \big( \gamma^{t+1} v_\pi(S_{t+1}) - \gamma^{t} v_\pi(S_t) \big) \right]$

The second sum telescopes: every term $\gamma^{t+1} v_\pi(S_{t+1})$ is cancelled by the next term's $-\gamma^{t+1} v_\pi(S_{t+1})$, and only $-v_\pi(S_0)$ survives (the terms at infinity vanish because of the discount). The first sum is the return of $\pi'$. Taking expectations,

$\displaystyle = \mathbb{E}_{\tau \sim \pi'}\big[ G(\tau) \big] - \mathbb{E}_{S_0 \sim p_0}\big[ v_\pi(S_0) \big] = J(\pi') - J(\pi)$

since the initial state has the same distribution under both policies. The lemma says something intuitive: to improve on $\pi$, the new policy should take, in the states it visits, actions that $\pi$ considers better than average. Policy iteration (Chapter 03) is the extreme case: the greedy policy takes the action with the largest advantage in every state, so every term is non-negative and the policy improves everywhere.

### The Surrogate Objective

The lemma is exact but, as it stands, useless: the expectation is over trajectories of $\pi'$, which we have not run. We rewrite it in terms of the states visited and the actions taken, with $d^{\pi'}$ the discounted distribution of states under $\pi'$:

$\displaystyle J(\pi') - J(\pi) = \frac{1}{1-\gamma}\, \mathbb{E}_{s \sim d^{\pi'}}\ \mathbb{E}_{a \sim \pi'(\cdot \mid s)}\big[ a_\pi(s,a) \big]$

Two expectations depend on $\pi'$, and we handle them differently. The **actions** are easy: the expectation over $a \sim \pi'$ can be rewritten as an expectation over $a \sim \pi$ with an **importance-sampling** correction, the ratio of the two probabilities,

$\displaystyle \mathbb{E}_{a \sim \pi'(\cdot \mid s)}\big[ a_\pi(s,a) \big] = \sum_a \pi'(a \mid s)\, a_\pi(s,a) = \sum_a \pi(a \mid s)\, \frac{\pi'(a \mid s)}{\pi(a \mid s)}\, a_\pi(s,a) = \mathbb{E}_{a \sim \pi(\cdot \mid s)}\left[ \frac{\pi'(a \mid s)}{\pi(a \mid s)}\, a_\pi(s,a) \right]$

which is exact, provided that the old policy gives positive probability to every action the new one may take, and involves only actions sampled from the old policy. The **states** are the problem: $d^{\pi'}$ is the distribution of the states that $\pi'$ would visit, and we only have states visited by $\pi$. Here we make an **approximation**: we replace $d^{\pi'}$ by $d^{\pi}$, arguing that if the two policies are close, they visit similar states. The result is the **surrogate objective**

$\displaystyle L_\pi(\pi') = \frac{1}{1-\gamma}\, \mathbb{E}_{s \sim d^{\pi},\ a \sim \pi(\cdot \mid s)}\left[ \frac{\pi'(a \mid s)}{\pi(a \mid s)}\, a_\pi(s,a) \right]$

(the factor $1/(1-\gamma)$ is the one of the lemma; in an implementation it is absorbed into the learning rate, as every constant is). The surrogate can be estimated entirely from a rollout of $\pi$: for every transition in the rollout, the ratio of the new to the old probability of the action taken, times the advantage estimate. Writing $\pi = \pi_{\theta_{\text{old}}}$ and $\pi' = \pi_\theta$, this is a function of $\theta$ that we can **optimise for several gradient steps on the same data**: the old policy is fixed, its probabilities were recorded at collection time, and only the numerator of the ratio changes.

The surrogate has two important properties. First, at $\theta = \theta_{\text{old}}$ the ratio is one and its gradient is the score function, so

$\displaystyle \nabla_\theta L_{\pi_{\theta_{\text{old}}}}(\pi_\theta)\Big|_{\theta = \theta_{\text{old}}} = \mathbb{E}\big[ \nabla_\theta \log \pi_\theta(a \mid s)\, a_\pi(s,a) \big]\Big|_{\theta = \theta_{\text{old}}} = \nabla_\theta J(\theta)\Big|_{\theta = \theta_{\text{old}}}$

the surrogate matches the true objective **to first order**: a small step along its gradient is exactly the policy gradient step of Chapter 08. Second, far from $\theta_{\text{old}}$ it may be badly wrong, because of the state-distribution approximation and because importance ratios with large values make the estimate unreliable. How far is too far is quantified by the following bound, proved by Schulman and colleagues in the [**TRPO**](./papers/2015%20-%20Trust%20Region%20Policy%20Optimization.pdf) paper (2015):

$\displaystyle J(\pi') \ \ge\ J(\pi) + L_\pi(\pi') - C \max_s D_{\text{KL}}\big( \pi(\cdot \mid s)\ \big\|\ \pi'(\cdot \mid s) \big)$

where the constant $C$ depends on $\gamma$ and on the largest advantage. The true improvement is at least the surrogate improvement minus a penalty proportional to the **largest KL divergence** between the two policies over the states. Maximising the right-hand side therefore guarantees improvement (a *minorisation-maximisation* argument, the same one behind the EM algorithm), and the penalty term says precisely how the step size should be controlled: not in parameter space, but in **KL divergence between policies**.

## Trust Region Policy Optimization (TRPO)

The bound suggests maximising the surrogate minus the KL penalty. In practice the constant $C$ is far too large (it makes the steps tiny), so TRPO turns the penalty into a **constraint**: maximise the surrogate subject to the average KL divergence staying below a threshold $\delta$, the radius of the **trust region** around the current policy,

$\displaystyle \max_\theta\ L_{\pi_{\theta_{\text{old}}}}(\pi_\theta) \quad \text{subject to} \quad \mathbb{E}_{s \sim d^{\pi_{\theta_{\text{old}}}}}\Big[ D_{\text{KL}}\big( \pi_{\theta_{\text{old}}}(\cdot \mid s)\ \big\|\ \pi_\theta(\cdot \mid s) \big) \Big] \le \delta$

(the average KL replaces the maximum of the bound, which cannot be estimated from samples). The constrained problem is solved approximately: the surrogate is replaced by its first-order Taylor expansion around $\theta_{\text{old}}$, which is just the policy gradient $g$, and the KL constraint by its second-order expansion, $\tfrac{1}{2} (\theta - \theta_{\text{old}})^\top F (\theta - \theta_{\text{old}}) \le \delta$, where $F$ is the Hessian of the KL divergence, known as the **Fisher information matrix**. The solution is the **natural gradient** step

$\displaystyle \theta = \theta_{\text{old}} + \sqrt{\frac{2\delta}{g^\top F^{-1} g}}\ F^{-1} g$

a gradient step rescaled by the inverse Fisher matrix, so that the step is large in directions in which the policy is insensitive to the parameters and small in directions in which it is sensitive: precisely the correction that the sigmoid example at the beginning of the chapter called for. Since $F$ has as many rows as the network has parameters, it is never formed explicitly: $F^{-1} g$ is computed by the conjugate gradient method, using only Fisher-vector products, and the result is checked by a line search that verifies that the surrogate actually improved and the KL constraint actually holds.

TRPO works, and it was one of the first methods to train neural network policies reliably on continuous control benchmarks. But it is **complicated**, with a second-order optimisation step inside every update, it is **expensive**, with several extra forward and backward passes per update, and it is **inflexible**: the conjugate gradient machinery is hard to combine with architectures that share parameters between actor and critic, with recurrent networks, or with stochastic layers such as dropout. The question that PPO answers is whether the trust region can be obtained with a first-order method.

## Proximal Policy Optimization (PPO)

### The Clipped Surrogate Objective

PPO keeps the surrogate objective and the idea that the policy should not move too far, but enforces the latter inside the objective itself, through the probability ratio

$\displaystyle r_t(\theta) = \frac{\pi_\theta(A_t \mid S_t)}{\pi_{\theta_{\text{old}}}(A_t \mid S_t)}$

which is $1$ for the old policy, larger than $1$ if the new policy makes the action taken more likely, and smaller if it makes it less likely. The surrogate is $\mathbb{E}[r_t(\theta) \hat{A}_t]$, with $\hat{A}_t$ the advantage estimate from the rollout. Maximising it without constraints would push $r_t$ to infinity where $\hat{A}_t > 0$ and to zero where $\hat{A}_t < 0$. PPO replaces it with the **clipped surrogate**

$\displaystyle L^{\text{CLIP}}(\theta) = \mathbb{E}_t \Big[ \min\Big( r_t(\theta)\, \hat{A}_t,\ \text{clip}\big( r_t(\theta),\ 1-\epsilon,\ 1+\epsilon \big)\, \hat{A}_t \Big) \Big]$

with $\epsilon$ a small constant, typically $0.1$ or $0.2$. To see what this does, consider the two cases separately:

| Advantage | Unclipped term | Clipped term | Minimum | Effect |
|---|---|---|---|---|
| $\hat{A}_t > 0$ (good action) | $r_t \hat{A}_t$, increasing in $r_t$ | $\text{clip}(r_t, 1-\epsilon, 1+\epsilon)\, \hat{A}_t$, flat outside the range | $r_t \hat{A}_t$ for $r_t \le 1+\epsilon$, then flat at $(1+\epsilon) \hat{A}_t$ | the action is made more likely, but **once $r_t$ exceeds $1+\epsilon$ the gradient is zero**: no incentive to go further |
| $\hat{A}_t < 0$ (bad action) | $r_t \hat{A}_t$, decreasing in $r_t$ | $\text{clip}(r_t, 1-\epsilon, 1+\epsilon)\, \hat{A}_t$, flat outside the range | $r_t \hat{A}_t$ for $r_t \ge 1-\epsilon$, then flat at $(1-\epsilon) \hat{A}_t$ | the action is made less likely, but **once $r_t$ drops below $1-\epsilon$ the gradient is zero** |

The minimum makes the objective a **pessimistic lower bound** of the unclipped surrogate: whenever the ratio moves in the direction that the advantage asks for, beyond $1 \pm \epsilon$, the objective stops rewarding it. Whenever the ratio moves in the wrong direction (a good action made less likely, or a bad action made more likely), the unclipped term is the smaller one and the gradient is kept: PPO can always undo a bad move but never gains from an excessive one. The clipped objective is not a trust region in the strict sense, since nothing prevents the ratio from leaving $[1-\epsilon, 1+\epsilon]$, but it removes the *incentive* to leave it, which in practice keeps the policy change small. The paper also proposes an alternative, **PPO-penalty**, which adds $-\beta_{\text{KL}} D_{\text{KL}}$ to the surrogate and adapts the coefficient so that the measured KL stays near a target; the clipped version is simpler and performs at least as well, and is the one everybody uses.

### Reusing the Rollout

The clipped objective is what makes it possible to take **several gradient steps on the same rollout**. After collecting $K \times n$ transitions with the current policy, exactly as in A2C, we record the log-probabilities of the actions taken and the GAE advantages, and then run **several epochs** of mini-batch gradient ascent on $L^{\text{CLIP}}$ (plus the value loss and the entropy bonus), typically $3$ to $10$ epochs with mini-batches of a few hundred transitions. During the first step the ratios are all one and the update is the ordinary policy gradient step; during the following steps the ratios drift away from one, and the clipping gradually switches off the contribution of the transitions whose probability has already changed enough. The number of epochs and $\epsilon$ together determine how far the policy moves per rollout. This is where the sample efficiency comes from: A2C performs one gradient step per rollout, PPO performs, say, $4$ epochs $\times$ $4$ mini-batches $= 16$ steps, so the same data move the policy much further, and safely.

A useful diagnostic is the **approximate KL divergence** between the old and the new policy, estimated from the ratios of the mini-batch as $\mathbb{E}_t[(r_t - 1) - \log r_t]$, an unbiased, always non-negative estimator of the KL divergence. If it grows beyond a few hundredths during the epochs, the policy is moving too far and the number of epochs or the learning rate should be reduced; some implementations stop the epochs early when a threshold is exceeded.

### The Full Objective

The complete PPO loss combines the clipped surrogate, the value loss and the entropy bonus:

$\displaystyle \mathcal{L}(\theta, \phi) = - L^{\text{CLIP}}(\theta) + c_v\, \mathbb{E}_t\Big[ \big( y_t - V(S_t;\phi) \big)^2 \Big] - \beta\, \mathbb{E}_t\Big[ H\big( \pi_\theta(\cdot \mid S_t) \big) \Big]$

where the critic targets $y_t$ are the GAE returns of Chapter 09, the coefficient $c_v$ matters only when actor and critic share parameters, and $\beta$ is the entropy coefficient. With separate networks, as in our implementation, the policy and value parts are optimised by two optimizers as before. Two further details are standard in PPO implementations and we adopt them: the advantages are **normalised** within each mini-batch (Exercise 9.8), and the gradients are **clipped** in norm.

### Algorithm

1. Collect a rollout of $n$ steps in $K$ parallel environments with the current policy $\pi_{\theta_{\text{old}}}$; record states, actions, rewards, terminal flags, the critic's values and the log-probabilities of the actions taken
2. Compute the GAE advantages $\hat{A}_t$ and the critic targets $y_t = \hat{A}_t + V(S_t;\phi)$
3. For $E$ epochs: shuffle the $K \times n$ transitions into mini-batches, and for each mini-batch compute the ratios $r_t(\theta)$ with the current parameters, the clipped surrogate, the value loss and the entropy, and take one gradient step on each network
4. Discard the rollout and repeat from step 1

Compared with A2C the only differences are in steps 1 and 3: the rollout must also store the states, the actions and the old log-probabilities, because the policy is re-evaluated on them, and the single update becomes a loop of epochs and mini-batches with the clipped objective.

## Implementation

We start again from Cart-Pole, so that we can compare PPO with A2C under the same conditions, and reuse the networks and helper functions of Chapter 09:

In [ ]:
import gymnasium

cart_pole = gymnasium.make('CartPole-v1', render_mode="rgb_array")

print('State Space: ', cart_pole.observation_space.shape)
print('Action Space: ', cart_pole.action_space.n)

In [ ]:
import torch
import numpy as np
import random

# set the seeds for reproducibility of results
seed = 34

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

cart_pole.reset(seed=seed)
cart_pole.action_space.seed(seed);

In [ ]:
def create_policy_network(env, first_hidden_layer=128, second_hidden_layer=64):

    # size of the state vector and number of actions
    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n

    # a fully connected network with two hidden layers
    dnn = torch.nn.Sequential(
        torch.nn.Linear(state_size, first_hidden_layer),
        torch.nn.ReLU(),
        torch.nn.Linear(first_hidden_layer, second_hidden_layer),
        torch.nn.ReLU(),
        # output layer: one preference (logit) per action
        torch.nn.Linear(second_hidden_layer, action_size),
    )

    return dnn


def create_value_network(env, first_hidden_layer=128, second_hidden_layer=64):

    # size of the state vector
    state_size = env.observation_space.shape[0]

    # a fully connected network with two hidden layers
    dnn = torch.nn.Sequential(
        torch.nn.Linear(state_size, first_hidden_layer),
        torch.nn.ReLU(),
        torch.nn.Linear(first_hidden_layer, second_hidden_layer),
        torch.nn.ReLU(),
        # output layer: the estimated value of the state
        torch.nn.Linear(second_hidden_layer, 1),
    )

    return dnn


pi = create_policy_network(cart_pole)
v = create_value_network(cart_pole)

The action distribution, the evaluation policies and the evaluation function are those of Chapter 09. The distribution function will be extended to continuous actions in the last section of the chapter; for now the network's output is interpreted as the logits of a categorical distribution:

In [ ]:
def policy_distribution(states):

    # convert one state, or a batch of states, into a tensor
    states = torch.as_tensor(np.asarray(states), dtype=torch.float32)

    # the softmax over the preferences defines the action distribution
    return torch.distributions.Categorical(logits=pi(states))


def stochastic_pi(state):

    # sample an action from the policy, as during training
    with torch.no_grad():
        action = policy_distribution(state).sample()

    return int(action.item())


def greedy_pi(state):

    # select the most probable action
    with torch.no_grad():
        preferences = pi(torch.as_tensor(state, dtype=torch.float32))

    return int(torch.argmax(preferences).item())

In [ ]:
def evaluate(env, policy, episodes=1):

    # collect total rewards per episode
    rewards = []

    for episode in range(episodes):

        # reset the environment
        state, _ = env.reset()
        done = False
        total_reward = 0.0

        # run one episode
        while not done:
            action = policy(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated

        rewards.append(total_reward)

    # return the average reward over episodes
    return float(np.mean(rewards))

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

def show_policy(env, policy, n_episodes=1, max_steps=500):

    for episode in range(n_episodes):

        # reset the environment
        state, _ = env.reset()
        done = False
        step = 0

        # run the episode until done or max_steps
        while not done and step < max_steps:

            # make a step
            action = policy(state)
            state, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # render the environment
            clear_output(wait=True)
            plt.imshow(env.render())
            plt.axis("off")
            plt.title(f"Episode {episode+1}, Step {step}")
            plt.show()
            plt.close()

            step += 1

The parallel environments are created as in Chapter 09:

In [ ]:
def create_environments(env, n_envs, seed):

    # K independent copies of the environment, each with its own seed
    envs = [gymnasium.make(env.spec.id) for _ in range(n_envs)]

    states = []
    for i, e in enumerate(envs):
        state, _ = e.reset(seed=seed + i)
        e.action_space.seed(seed + i)
        states.append(state)

    return envs, np.array(states, dtype=np.float32)

### Collecting a Rollout

The rollout of PPO differs from that of A2C in what it stores. Since the policy will be re-evaluated on the collected transitions, we must keep the **states** and the **actions**, and the **log-probabilities of the old policy**, which are the denominators of the ratios; these are recorded as plain numbers, without gradient, and so are the values, which are only needed to compute the advantages. The handling of episode boundaries, termination versus truncation, is exactly as in Chapter 09. A small helper converts the sampled action into what the environment expects, an integer for a discrete action space, a clipped array for a continuous one:

In [ ]:
def environment_action(env, action):

    # discrete action: an integer; continuous action: an array clipped to the bounds
    if isinstance(env.action_space, gymnasium.spaces.Discrete):
        return int(action)

    return np.clip(action.numpy(), env.action_space.low, env.action_space.high)


def collect_rollout(envs, states, n_steps, gamma=0.99):

    n_envs = len(envs)

    # quantities recorded along the rollout, one row per step, one column per environment
    rollout_states, rollout_actions, logpas, values = [], [], [], []
    rewards = np.zeros((n_steps, n_envs), dtype=np.float32)
    dones = np.zeros((n_steps, n_envs), dtype=np.float32)

    for t in range(n_steps):

        # one batched forward pass for all the environments, without gradient
        with torch.no_grad():
            dist = policy_distribution(states)
            actions = dist.sample()

            rollout_states.append(states.copy())
            rollout_actions.append(actions)
            logpas.append(dist.log_prob(actions))
            values.append(v(torch.as_tensor(states)).squeeze(-1).numpy())

        # step every environment with its own action
        next_states = np.zeros_like(states)

        for i, env in enumerate(envs):
            next_state, reward, terminated, truncated, _ = env.step(environment_action(env, actions[i]))

            # a truncated episode has a future: bootstrap on the state reached
            if truncated and not terminated:
                with torch.no_grad():
                    reward += gamma * v(torch.as_tensor(next_state, dtype=torch.float32)).item()

            # at the end of an episode, start a new one
            if terminated or truncated:
                next_state, _ = env.reset()

            rewards[t, i] = reward
            dones[t, i] = float(terminated or truncated)
            next_states[i] = next_state

        states = next_states

    # the critic's estimate of the last states, to bootstrap the unfinished episodes
    with torch.no_grad():
        last_values = v(torch.as_tensor(states)).squeeze(-1).numpy()

    rollout = {
        "states": np.stack(rollout_states),
        "actions": torch.stack(rollout_actions),
        "logpas": torch.stack(logpas),
        "values": np.stack(values),
        "rewards": rewards,
        "dones": dones,
        "last_values": last_values,
    }

    return rollout, states

The GAE advantages and the critic targets are computed as in Chapter 09; this time the function returns both:

In [ ]:
def compute_gae(rewards, dones, values, last_values, gamma=0.99, lam=0.95):

    advantages = np.zeros_like(rewards)
    future_advantage = np.zeros_like(last_values)
    next_values = last_values

    for t in reversed(range(len(rewards))):

        # one-step TD error, with no future value after a terminal state
        td_error = rewards[t] + gamma * (1.0 - dones[t]) * next_values - values[t]

        # A_t = delta_t + gamma * lambda * (1 - d_{t+1}) * A_{t+1}
        future_advantage = td_error + gamma * lam * (1.0 - dones[t]) * future_advantage
        advantages[t] = future_advantage

        next_values = values[t]

    # the critic target is the advantage plus the current value (the lambda-return)
    returns = advantages + values

    return torch.as_tensor(advantages), torch.as_tensor(returns)

### Loss Functions

The optimization step is where PPO differs from A2C. The rollout is flattened into a batch of $K \times n$ transitions, and for a number of epochs it is shuffled and split into mini-batches. For each mini-batch the policy is re-evaluated on the stored states to obtain the new log-probabilities of the stored actions; the ratio is the exponential of the difference between new and old log-probabilities; the clipped surrogate is the minimum of the unclipped and clipped terms, with the advantages normalised within the mini-batch; the entropy bonus is added, and the critic regresses towards the stored targets. After the epochs, the function re-evaluates the policy on the whole rollout and returns the approximate KL divergence between the old and the new policy, as a diagnostic:

In [ ]:
def optimize(rollout, advantages, returns, epochs=4, minibatch_size=256, epsilon=0.2, beta=0.01, max_gradient_norm=0.5):

    # flatten the rollout into a batch of K x n transitions
    n_steps, n_envs = rollout["rewards"].shape
    batch_size = n_steps * n_envs

    states = torch.as_tensor(rollout["states"]).reshape(batch_size, -1)
    actions = rollout["actions"].reshape(batch_size, *rollout["actions"].shape[2:])
    old_logpas = rollout["logpas"].reshape(batch_size)
    advantages = advantages.reshape(batch_size)
    returns = returns.reshape(batch_size)

    for epoch in range(epochs):

        # shuffle the transitions and split them into mini-batches
        permutation = torch.randperm(batch_size)

        for start in range(0, batch_size, minibatch_size):

            indices = permutation[start:start + minibatch_size]

            # re-evaluate the current policy on the stored states and actions
            dist = policy_distribution(states[indices])
            new_logpas = dist.log_prob(actions[indices])
            entropy = dist.entropy()

            # probability ratios between the current and the old policy
            ratios = (new_logpas - old_logpas[indices]).exp()

            # advantages normalised within the mini-batch
            minibatch_advantages = advantages[indices]
            minibatch_advantages = (minibatch_advantages - minibatch_advantages.mean()) / (minibatch_advantages.std() + 1e-8)

            # clipped surrogate objective: the minimum of the unclipped and clipped terms
            unclipped = ratios * minibatch_advantages
            clipped = ratios.clamp(1.0 - epsilon, 1.0 + epsilon) * minibatch_advantages
            surrogate = torch.min(unclipped, clipped).mean()

            # policy loss: minus the surrogate, minus the entropy bonus
            policy_loss = -(surrogate + beta * entropy.mean())

            # value loss: regression of V(s) towards the GAE returns
            value_loss = (returns[indices] - v(states[indices]).squeeze(-1)).pow(2).mean()

            # update the actor
            policy_optimizer.zero_grad()
            policy_loss.backward()
            torch.nn.utils.clip_grad_norm_(pi.parameters(), max_gradient_norm)
            policy_optimizer.step()

            # update the critic
            value_optimizer.zero_grad()
            value_loss.backward()
            torch.nn.utils.clip_grad_norm_(v.parameters(), max_gradient_norm)
            value_optimizer.step()

    # approximate KL divergence between the old and the new policy over the whole rollout (diagnostic)
    with torch.no_grad():
        ratios = (policy_distribution(states).log_prob(actions) - old_logpas).exp()
        approximate_kl = ((ratios - 1.0) - ratios.log()).mean().item()

    return approximate_kl

### Training Loop

The training loop is that of A2C with the new rollout and the new optimization step. The rollouts are longer than in A2C ($n = 128$ steps in $K = 8$ environments, a batch of $1024$ transitions), because each rollout is now used for several epochs, and the number of updates per environment step is correspondingly smaller:

In [ ]:
def ppo(env, max_steps, eval_every=2000, n_envs=8, n_steps=128, epochs=4, minibatch_size=256,
        gamma=0.99, lam=0.95, epsilon=0.2, beta=0.01, eval_policy=None):

    # the policy used for evaluation (greedy by default)
    eval_policy = greedy_pi if eval_policy is None else eval_policy

    # the K environments and a separate, seeded environment for evaluation
    envs, states = create_environments(env, n_envs, seed=int(env.np_random.integers(1_000_000)))
    eval_env = gymnasium.make(env.spec.id)
    eval_env.reset(seed=int(env.np_random.integers(1_000_000)))

    scores = []
    total_steps = 0
    next_evaluation = eval_every

    while total_steps < max_steps:

        # collect a rollout of n steps in the K environments with the current policy
        rollout, states = collect_rollout(envs, states, n_steps, gamma)
        total_steps += n_steps * n_envs

        # GAE advantages and critic targets
        advantages, returns = compute_gae(rollout["rewards"], rollout["dones"], rollout["values"], rollout["last_values"], gamma, lam)

        # several epochs of mini-batch updates on the same rollout
        approximate_kl = optimize(rollout, advantages, returns, epochs, minibatch_size, epsilon, beta)

        # evaluate the policy at regular intervals
        if total_steps >= next_evaluation:
            score = evaluate(eval_env, eval_policy, episodes=10)
            scores.append(score)
            next_evaluation += eval_every
            print(f"Step {total_steps:07}, score {score:06.1f}, approximate KL {approximate_kl:.4f}", end="\r", flush=True)

    return scores

### Solve Cart-Pole

We create the optimizers, with the usual larger learning rate for the critic, and train with the same budget of environment steps used for A2C in Chapter 09:

In [ ]:
actor_learning_rate = 0.0003
critic_learning_rate = 0.001

policy_optimizer = torch.optim.Adam(pi.parameters(), lr=actor_learning_rate)
value_optimizer = torch.optim.Adam(v.parameters(), lr=critic_learning_rate)

scores = ppo(cart_pole, max_steps=150_000)

In [ ]:
agent_score = evaluate(cart_pole, greedy_pi, episodes=10)
print("Agent score", agent_score)

In [ ]:
show_policy(cart_pole, greedy_pi, n_episodes=1, max_steps=500)

For the controlled experiment we reuse the step-based experiment and plotting functions of Chapter 09:

In [ ]:
import joblib
from pathlib import Path

def experiment(
    env,
    train,
    setup,
    max_steps,
    name,
    eval_every=2000,
    seeds=(12, 34, 56, 78, 90),
    smoothing_window=5,
):

    results_path = Path("temp") / f"{name}_results.joblib"

    # load saved results if they already exist
    if results_path.exists():
        experiment_results = joblib.load(results_path)

        n_points = len(experiment_results["mean_score"])
        final_mean = experiment_results["mean_score"][-1]
        final_min = experiment_results["min_score"][-1]
        final_max = experiment_results["max_score"][-1]

        print(f"Loaded saved results from {results_path}")
        print(f"Smoothed points: {n_points}")
        print(f"Smoothing window: {experiment_results['smoothing_window']}")
        print(f"Final score: mean={final_mean:.1f}, range=[{final_min:.1f}, {final_max:.1f}]")

        return experiment_results

    results = []

    for seed in seeds:
        print(f"Experiment seed: {seed}")

        # set seeds for reproducibility
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        env.reset(seed=seed)
        env.action_space.seed(seed)

        # initialize networks and optimizers
        setup()

        # train the agent
        scores = train(env, max_steps=max_steps, eval_every=eval_every)

        # smooth the learning curve
        kernel = np.ones(smoothing_window) / smoothing_window
        scores = np.convolve(scores, kernel, mode="valid")

        results.append(scores)
        print()

    results = np.array(results)

    experiment_results = {
        "max_score": np.max(results, axis=0),
        "min_score": np.min(results, axis=0),
        "mean_score": np.mean(results, axis=0),
        "smoothing_window": smoothing_window,
        "eval_every": eval_every,
        "seeds": seeds,
        "max_steps": max_steps,
    }

    results_path.parent.mkdir(exist_ok=True)
    joblib.dump(experiment_results, results_path)

    return experiment_results

In [ ]:
def steps_axis(results):
    # the smoothed curves start after the first smoothing window
    offset = results.get("smoothing_window", 1)
    return (np.arange(len(results["mean_score"])) + offset) * results.get("eval_every", 1)


def plot_comparison(
    first_results,
    second_results,
    first_label="A",
    second_label="B",
    first_color="blue",
    second_color="red",
):

    first_steps = steps_axis(first_results)
    second_steps = steps_axis(second_results)

    plt.figure(figsize=(12, 6))
    plt.title(f"{second_label} vs {first_label}")
    plt.xlabel("Environment steps")
    plt.ylabel("Score")

    # plot the first algorithm
    plt.plot(first_steps, first_results["mean_score"], color=first_color, linewidth=2, label=f"{first_label} mean")
    plt.fill_between(first_steps, first_results["min_score"], first_results["max_score"], color=first_color, alpha=0.20, label=f"{first_label} range")

    # plot the second algorithm
    plt.plot(second_steps, second_results["mean_score"], color=second_color, linewidth=2, label=f"{second_label} mean")
    plt.fill_between(second_steps, second_results["min_score"], second_results["max_score"], color=second_color, alpha=0.20, label=f"{second_label} range")

    plt.legend()
    plt.show()

In [ ]:
# define the experiment setup and hyperparameters
gamma = 0.99                  # discount factor
lam = 0.95                    # GAE parameter
epsilon = 0.2                 # clipping range
beta = 0.01                   # entropy coefficient
epochs = 4                    # optimization epochs per rollout
minibatch_size = 256          # transitions per mini-batch
n_envs = 8                    # number of parallel environments (K)
n_steps = 128                 # steps per rollout (n)
actor_learning_rate = 0.0003  # step size of the actor's optimizer
critic_learning_rate = 0.001  # step size of the critic's optimizer
first_hidden_layer = 128      # size of the first hidden layer
second_hidden_layer = 64      # size of the second hidden layer

def setup_ppo():
    global pi, v, policy_optimizer, value_optimizer
    pi = create_policy_network(cart_pole, first_hidden_layer, second_hidden_layer)
    v = create_value_network(cart_pole, first_hidden_layer, second_hidden_layer)
    policy_optimizer = torch.optim.Adam(pi.parameters(), lr=actor_learning_rate)
    value_optimizer = torch.optim.Adam(v.parameters(), lr=critic_learning_rate)

def train_ppo(env, max_steps, eval_every):
    return ppo(env, max_steps, eval_every, n_envs, n_steps, epochs, minibatch_size, gamma, lam, epsilon, beta)

# run the experiment
ppo_results = experiment(
    env=cart_pole,
    train=train_ppo,
    setup=setup_ppo,
    max_steps=150_000,
    name="ppo",
)

The A2C experiment of Chapter 09 used the same budget, the same evaluation interval and the same networks (with RMSprop instead of Adam as optimizer, a minor difference), so its saved results, if available in the `temp` folder of Chapter 09, can be plotted together with the PPO curve:

In [ ]:
a2c_results_path = Path("../09 - Actor-Critic Methods/temp/a2c_gae_results.joblib")

if a2c_results_path.exists():
    a2c_results = joblib.load(a2c_results_path)
    plot_comparison(a2c_results, ppo_results, first_label="A2C with GAE", second_label="PPO", first_color="green", second_color="purple")
else:
    print("A2C results not found: run the experiments of Chapter 09 first.")

On Cart-Pole PPO reaches the maximum score much earlier than A2C, and holds it: the range across seeds is narrow and the curve, once at the top, stays there. Both algorithms use the same kind of data, the same networks and the same advantage estimates; PPO extracts more from each rollout, sixteen gradient steps on each batch of $1024$ transitions where A2C took one step on each batch of $160$, and the clipped objective prevents those steps from moving the policy too far. The printed approximate KL divergence stays around a hundredth or less throughout: the policy moves a little at every update, and never jumps.

## Solve Lunar Lander

We can now return to the discrete Lunar Lander, on which A2C, in Chapter 09, learned to hover and drift towards the pad but oscillated and did not land reliably within a million steps. We apply PPO with the same budget and, as in Chapter 09, evaluate the stochastic policy:

In [ ]:
lunar_lander = gymnasium.make('LunarLander-v3', render_mode="rgb_array")

print('State Space: ', lunar_lander.observation_space.shape)
print('Action Space: ', lunar_lander.action_space.n)

In [ ]:
# set the seeds for reproducibility of results
seed = 34

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

lunar_lander.reset(seed=seed)
lunar_lander.action_space.seed(seed)

# fresh networks and optimizers for the new environment
pi = create_policy_network(lunar_lander, first_hidden_layer=128, second_hidden_layer=128)
v = create_value_network(lunar_lander, first_hidden_layer=128, second_hidden_layer=128)
policy_optimizer = torch.optim.Adam(pi.parameters(), lr=actor_learning_rate)
value_optimizer = torch.optim.Adam(v.parameters(), lr=critic_learning_rate)

random_score = evaluate(lunar_lander, lambda state: lunar_lander.action_space.sample(), episodes=100)
print("Random score: ", random_score)

In [ ]:
scores = ppo(lunar_lander, max_steps=1_000_000, eval_every=50_000, n_envs=16, n_steps=128, eval_policy=stochastic_pi)

In [ ]:
eval_every = 50_000

plt.figure(figsize=(12, 6))
plt.title("PPO on Lunar Lander (stochastic policy)")
plt.xlabel("Environment steps")
plt.ylabel("Score")
plt.plot((np.arange(len(scores)) + 1) * eval_every, scores, color="purple", linewidth=2)
plt.axhline(200, color="gray", linestyle="--", linewidth=1, label="Solved")
plt.legend()
plt.show()

In [ ]:
stochastic_score = evaluate(lunar_lander, stochastic_pi, episodes=10)
greedy_score = evaluate(lunar_lander, greedy_pi, episodes=10)

print("Random score", random_score)
print("Agent score (stochastic policy)", stochastic_score)
print("Agent score (greedy policy)", greedy_score)

In [ ]:
show_policy(lunar_lander, greedy_pi, n_episodes=1, max_steps=1000)

PPO does what A2C could not: the score climbs steadily, with far smaller oscillations than in Chapter 09, and crosses the solved threshold at about half of the budget, staying above it afterwards; the greedy policy, at the end, is as good as or better than the stochastic one, a sign that the policy has become confident where it should be. The data are the same kind of data that A2C used, collected on-policy by parallel environments, and the advantages are the same GAE estimates. What changed is the update: many small, safe steps per rollout instead of one unconstrained step. This is the step-size problem of Chapter 08 and Chapter 09, finally solved.

## Continuous Actions

Everything in this chapter applies to continuous actions as well, with the Gaussian policy of Chapter 10 in place of the softmax. The policy network outputs the mean of the action; the logarithm of the standard deviation is a separate parameter vector, independent of the state, which the optimizer adjusts like any other weight (a state-dependent standard deviation is also possible, but the state-independent choice is simpler and standard in PPO implementations). Following common practice for Gaussian policies, the mean network uses $\tanh$ activations, whose bounded output keeps the mean from drifting far outside the action range early in training. The log-probability of an action is the sum over its components of the Gaussian log-densities, the entropy is the sum of the Gaussian entropies, and the ratio $r_t(\theta)$ is the ratio of densities. We write the policy as a module that declares its log-standard-deviation parameter, so that the distribution function can recognise it:

We will apply PPO to the continuous Lunar Lander of Chapter 10:

In [ ]:
lunar_lander = gymnasium.make('LunarLanderContinuous-v3', render_mode="rgb_array")

print('State Space: ', lunar_lander.observation_space.shape)
print('Action Space: ', lunar_lander.action_space)

In [ ]:
class GaussianPolicy(torch.nn.Module):

    def __init__(self, env, first_hidden_layer=128, second_hidden_layer=128):
        super().__init__()

        state_size = env.observation_space.shape[0]
        action_size = env.action_space.shape[0]

        # the mean of the action is computed from the state
        self.mean_network = torch.nn.Sequential(
            torch.nn.Linear(state_size, first_hidden_layer),
            torch.nn.Tanh(),
            torch.nn.Linear(first_hidden_layer, second_hidden_layer),
            torch.nn.Tanh(),
            torch.nn.Linear(second_hidden_layer, action_size),
        )

        # the log standard deviation is a state-independent parameter
        self.log_std = torch.nn.Parameter(torch.zeros(action_size))

    def forward(self, state):
        return self.mean_network(state)

The distribution function returns a Gaussian with independent components when the policy is Gaussian, so that `log_prob` and `entropy` sum over the action dimensions, and the evaluation policies return clipped arrays:

In [ ]:
def policy_distribution(states):

    # convert one state, or a batch of states, into a tensor
    states = torch.as_tensor(np.asarray(states), dtype=torch.float32)

    # Gaussian policy: independent components with the network's mean and the learned standard deviation
    if hasattr(pi, "log_std"):
        normal = torch.distributions.Normal(pi(states), pi.log_std.exp())
        return torch.distributions.Independent(normal, 1)

    # otherwise, the softmax over the preferences
    return torch.distributions.Categorical(logits=pi(states))


def stochastic_pi(state):

    # sample an action from the policy and clip it to the bounds
    with torch.no_grad():
        action = policy_distribution(state).sample()

    return np.clip(action.numpy(), lunar_lander.action_space.low, lunar_lander.action_space.high)


def greedy_pi(state):

    # the mean action, clipped to the bounds
    with torch.no_grad():
        action = pi(torch.as_tensor(state, dtype=torch.float32))

    return np.clip(action.numpy(), lunar_lander.action_space.low, lunar_lander.action_space.high)

Nothing else changes: the rollout, the advantages, the clipped objective and the training loop are the same functions. We train with no entropy bonus, since the learned standard deviation provides the exploration, and evaluate the mean action:

In [ ]:
# set the seeds for reproducibility of results
seed = 34

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

lunar_lander.reset(seed=seed)
lunar_lander.action_space.seed(seed)

# fresh networks and optimizers for the new environment
pi = GaussianPolicy(lunar_lander)
v = create_value_network(lunar_lander, first_hidden_layer=128, second_hidden_layer=128)
policy_optimizer = torch.optim.Adam(pi.parameters(), lr=actor_learning_rate)
value_optimizer = torch.optim.Adam(v.parameters(), lr=critic_learning_rate)

random_score = evaluate(lunar_lander, lambda state: lunar_lander.action_space.sample(), episodes=100)
print("Random score: ", random_score)

In [ ]:
scores = ppo(lunar_lander, max_steps=1_000_000, eval_every=50_000, n_envs=16, n_steps=128, beta=0.0)

In [ ]:
eval_every = 50_000

plt.figure(figsize=(12, 6))
plt.title("PPO on Lunar Lander (continuous)")
plt.xlabel("Environment steps")
plt.ylabel("Score")
plt.plot((np.arange(len(scores)) + 1) * eval_every, scores, color="purple", linewidth=2)
plt.axhline(200, color="gray", linestyle="--", linewidth=1, label="Solved")
plt.legend()
plt.show()

In [ ]:
agent_score = evaluate(lunar_lander, greedy_pi, episodes=10)

print("Random score", random_score)
print("Agent score", agent_score)
print("Learned standard deviation", pi.log_std.exp().detach().numpy())

In [ ]:
show_policy(lunar_lander, greedy_pi, n_episodes=1, max_steps=1000)

PPO solves the continuous task as well, and the learned standard deviation, which started at $1$, has shrunk during training: the agent explored widely at the beginning and became more precise as the critic improved, an adaptive exploration of the same kind that SAC obtained with its automatic temperature. The comparison with Chapter 10 is instructive. SAC reached comparable scores in roughly a third to a half of the environment steps, because it replays every transition many times; PPO needs more steps, but each of them costs a fraction of the computation, since the thirty-two gradient steps per rollout (four epochs of eight mini-batches) are spread over $2048$ transitions, where SAC performs one gradient step on three networks and the temperature for every single transition. In wall-clock time, on a simulator this fast, PPO is the quicker of the two. This trade-off, sample efficiency against computational efficiency and simplicity, is the main criterion for choosing between the two families in practice: when environment interaction is cheap, as in simulation, PPO is usually preferred; when it is expensive, as with a physical robot, the off-policy methods of Chapter 10 are.

## Limitations and Outlook

PPO is the end point of the on-policy line that started with REINFORCE: a parameterised policy improved by gradient ascent, a critic that estimates advantages, parallel environments that decorrelate the data, and now an objective that allows several safe gradient steps per rollout. It is robust, simple to implement, and works with any action space and any network architecture, which is why it is the default choice in most applications, including the reinforcement learning stage of training large language models, where the "environment" is a reward model and the "actions" are tokens (Chapter 13). Its limitations are those of its design choices.

First, PPO is still **on-policy**: each rollout is used for a few epochs and discarded, so its sample efficiency, although much better than A2C's, remains well below that of the replay-based methods of Chapter 10. Second, the clipped objective is a **heuristic**, not a constraint: nothing prevents the ratios from leaving the clipping range, and in practice the policy can still move too far when the learning rate or the number of epochs is too large, which is why the approximate KL divergence must be monitored. Third, PPO is known to be sensitive to **implementation details** that the algorithm itself does not specify, such as advantage normalisation, value function clipping, learning rate schedules, observation and reward scaling, and the initialisation of the networks; two implementations that differ in these choices can perform very differently on the same task. Finally, like every method in this course so far, PPO assumes that the agent can **interact** with the environment to collect its own data. The next chapter asks what can be done when it cannot.